# Measuring throughput with Mininet

In this exercise you will:

1. build a small emulated network with **Mininet** and check that it works;
2. run the simple TCP/UDP sender and receiver in `apps/` on top of it (a tiny `iperf`);
3. change the network characteristics (bandwidth, latency, packet loss) and study how the measured throughput reacts.

> **Requirements:** this notebook must run on a Linux machine with Mininet installed, and Jupyter must run as **root**
> (Mininet creates network namespaces, virtual interfaces and traffic-control rules). See the `README.md`.

## 0. Setup

Mininet can be used from the command line (`sudo mn ...`) or, as we do here, from Python.
Each Mininet *host* is a normal process living in its own **network namespace**: it has its own interfaces,
IP addresses and routing table, but it **shares the file system** with your machine. This means the hosts
can run the scripts in `apps/` directly, and any change you make to those files is visible immediately.

In [ ]:
import os
import time

from mininet.net import Mininet
from mininet.topo import Topo
from mininet.node import OVSBridge
from mininet.link import TCLink
from mininet.log import setLogLevel
from mininet.clean import cleanup

assert os.geteuid() == 0, "Mininet needs root: start Jupyter with sudo"

setLogLevel("info")
# Remove leftovers from a previous (crashed) run
cleanup()

APPS = os.path.abspath("apps")
net = None

## 1. A simple two-host network

Our topology is the simplest possible one:

```
 h1 (10.0.0.1) ---- s1 ---- h2 (10.0.0.2)
     client       switch       server
```

* `h1` and `h2` are hosts, `s1` is an Ethernet switch (we use `OVSBridge`, a plain learning switch, so we don't need an SDN controller).
* The link `h1 - s1` is the one we will *shape* later: with `TCLink` Mininet uses Linux traffic control (`tc`)
  to impose a bandwidth (`bw`, in Mbit/s), a delay (`delay`, e.g. `'10ms'`) and a loss rate (`loss`, in %).
  These settings apply **in both directions** of that link. The `h2 - s1` link is left unconstrained.

The function below (re)builds the network with the given link characteristics. We will call it again every
time we want to change them.

In [ ]:
class TwoHostTopo(Topo):
    def build(self, bw=None, delay=None, loss=None):
        h1 = self.addHost("h1", ip="10.0.0.1/24")
        h2 = self.addHost("h2", ip="10.0.0.2/24")
        s1 = self.addSwitch("s1")
        # Shaped link (the "bottleneck")
        self.addLink(h1, s1, bw=bw, delay=delay, loss=loss)
        # Unconstrained link
        self.addLink(h2, s1)


def build_net(bw=None, delay=None, loss=None):
    # Stop the previous network (if any) and start a new one
    global net
    if net is not None:
        net.stop()
    topo = TwoHostTopo(bw=bw, delay=delay, loss=loss)
    net = Mininet(topo=topo, link=TCLink, switch=OVSBridge, controller=None)
    net.start()
    return net


build_net()
h1, h2 = net.get("h1", "h2")

### Testing connectivity

`h.cmd(...)` runs a shell command **inside** host `h` and returns its output.
Let's look at the interfaces of `h1` and ping `h2` from it.

In [ ]:
print(h1.cmd("ip -brief addr"))
print(h1.cmd("ping -c 3 10.0.0.2"))

In [ ]:
# Mininet also has a helper that pings between all pairs of hosts
net.pingAll()

**Question 1.1** — Note the RTT reported by `ping`. We have not added any delay yet: where does this time come from?

## 2. Running our applications on the network

The applications in `apps/` were written to run on `127.0.0.1` (the loopback interface of your machine).
Inside Mininet, the receiver runs on `h2` and the sender on `h1`, so **you need to change the IP addresses**:

* in `apps/tcp-receiver.py` (and `udp-receiver.py`), `localIP` must be the address of the server, `10.0.0.2`
  (alternatively `"0.0.0.0"`, meaning "all interfaces");
* in `apps/tcp-sender.py` (and `udp-sender.py`), `serverIP` must be `10.0.0.2`.

**Before going on**, also complete the last line of both receivers: compute and print the throughput in **Mbit/s**
from `total_size` (bytes) and the elapsed time.

Now start the receiver on `h2` **in the background** (`&`), writing its output to a log file.
We use `python3 -u` (unbuffered output) so that what it prints appears immediately in the log.

In [ ]:
h2.cmd(f"python3 -u {APPS}/tcp-receiver.py > /tmp/tcp-receiver.log 2>&1 &")
time.sleep(1)  # give the server time to start listening
print(h2.cmd("ss -ltn"))  # the server should be listening on port 20003

Then run the sender on `h1`. This call blocks until the sender is done.

In [ ]:
print(h1.cmd(f"python3 {APPS}/tcp-sender.py"))
time.sleep(1)
print(h2.cmd("cat /tmp/tcp-receiver.log"))

The receiver keeps running and waits for the next connection: you can run the sender again and a new line will
appear in the log. When you are done, stop it:

In [ ]:
h2.cmd("pkill -f tcp-receiver.py")

Since we will repeat this a lot, here is a helper that runs one full experiment (start receiver, run sender,
wait for the receiver's result, stop the receiver). It gives up after `timeout` seconds.

In [ ]:
def run_experiment(proto="tcp", timeout=30):
    log = f"/tmp/{proto}-receiver.log"
    h2.cmd(f"python3 -u {APPS}/{proto}-receiver.py > {log} 2>&1 &")
    time.sleep(1)
    print("sender  :", h1.cmd(f"python3 {APPS}/{proto}-sender.py").strip())
    # Wait for the receiver to print its result
    result = ""
    deadline = time.time() + timeout
    while not result and time.time() < deadline:
        time.sleep(0.5)
        result = h2.cmd(f"cat {log}").strip()
    print("receiver:", result or f"(no result after {timeout} s)")
    h2.cmd(f"pkill -f {proto}-receiver.py")


run_experiment("tcp")

**Question 2.1** — The sender and the receiver both measure a time. Are they the same? Which one would you trust
to compute the throughput, and why? (Hint: when does `sendall()` return? When does the receiver start its timer?)

**Question 2.2** — Run the same experiment several times. How much do the results vary? Try changing
`total_bytes` in `tcp-sender.py`: what amount of data gives a stable estimate?

## 3. Message size

The sender passes the data to the socket in messages of `msg_len` bytes (one `sendall()` call per message).

**Question 3.1** — Before trying: which message size do you expect to give the highest throughput, and why?

Now measure it: edit `msg_len` in `apps/tcp-sender.py` (e.g. 1, 10, 100, 1 000, 10 000, 100 000 bytes; keep
`total_bytes` large enough) and re-run the cell below after each change. Write down the results.

In [ ]:
run_experiment("tcp")

**Question 3.2** — How does throughput change with `msg_len`? Where is the time spent when messages are very small?
Why does the gain stop after a certain size?

**Question 3.3** — The receiver also has a parameter, `bufferSize`. Does it play a similar role?

**Question 3.4** — Repeat a few measurements with UDP (`run_experiment("udp")`, after updating `msg_len` in
`udp-sender.py`). What happens when `msg_len` is larger than ~1472 bytes (the Ethernet MTU of 1500 bytes minus the IP and UDP
headers)? And larger than 65 507 bytes?

In [ ]:
run_experiment("udp")

## 4. Changing bandwidth and latency

So far the network was "infinitely" fast: the throughput was limited only by the CPU of your machine.
Let's now rebuild the network with a constrained link. For example, a 10 Mbit/s link:

In [ ]:
build_net(bw=10)
h1, h2 = net.get("h1", "h2")
run_experiment("tcp")

And a link with 50 ms of delay (in each direction, so the RTT becomes ~100 ms). Check it with `ping` first:

In [ ]:
build_net(delay="50ms")
h1, h2 = net.get("h1", "h2")
print(h1.cmd("ping -c 3 10.0.0.2"))
run_experiment("tcp")

Now explore. Use the cell below, changing `bw` and `delay` (you can combine them), and `total_bytes` / `msg_len`
in the sender. Keep a small table of your results.

In [ ]:
build_net(bw=10, delay="20ms")
h1, h2 = net.get("h1", "h2")
run_experiment("tcp")

**Question 4.1** — With `bw=10` and no delay, how close is the measured throughput to 10 Mbit/s? Why is it not exactly 10?

**Question 4.2** — With a large delay and a small `total_bytes` (e.g. 100 KB), the measured throughput is much lower than
the link bandwidth, even though the link is idle. Why? What happens when you increase `total_bytes`?
(Think about the TCP handshake, *slow start*, and how many round trips the transfer needs.)

**Question 4.3** — Does `msg_len` still matter when the bottleneck is the network rather than the CPU?

**Question 4.4** — Run UDP on the 10 Mbit/s link. Compare the bytes sent and the bytes received. What does the UDP
sender do that the TCP sender does not (or vice versa)?

## 5. Packet loss (and why UDP may never finish)

Let's add random packet loss on the link: `loss=5` drops 5% of the packets, in each direction.

In [ ]:
build_net(loss=5)
h1, h2 = net.get("h1", "h2")
print(h1.cmd("ping -c 20 -i 0.2 10.0.0.2 | tail -2"))

First with TCP:

In [ ]:
run_experiment("tcp")

**Question 5.1** — Did the receiver get all the bytes? How did the throughput change compared to the lossless case?
Try `loss=1`, `loss=5`, `loss=10`, alone and combined with some delay.

Now with UDP. Remember that UDP has **no connection**: the receiver cannot see the sender closing its socket.
Our sender therefore tells the receiver that the transfer is over by sending a special `END` message, and the
receiver stops its timer when it gets it. This time we start the receiver **once** and run the sender several times,
without stopping the receiver in between:

In [ ]:
h2.cmd(f"python3 -u {APPS}/udp-receiver.py > /tmp/udp-receiver.log 2>&1 &")
time.sleep(1)
for i in range(5):
    print(f"run {i}:", h1.cmd(f"python3 {APPS}/udp-sender.py").strip())
    time.sleep(1)
print("--- receiver log ---")
print(h2.cmd("cat /tmp/udp-receiver.log"))

In [ ]:
h2.cmd("pkill -f udp-receiver.py")

**Question 5.2** — How many lines did the receiver print compared to the number of runs? Compare the bytes received
with the bytes sent. What happened when the `END` message was lost? (If it is not happening, increase `loss` and retry.)

**Question 5.3** — Modify the UDP applications so that the receiver always terminates. Some ideas:
send `END` several times, or use a timeout on the receiver socket (`s.settimeout(...)`, which raises
`socket.timeout`). What are the drawbacks of each approach?

**Question 5.4** — With UDP, the receiver knows how many bytes it got, but not how many were sent. How could you let
it compute the loss rate? (Hint: what could you put inside each message?)

## 6. Wrap-up

Answer briefly:

1. Why does ping show a non-zero RTT even without any configured delay? What does `delay="50ms"` on the `h1 - s1` link do to the RTT, and why?
2. Why does sending many tiny messages give a low throughput even on a very fast network?
3. On a link with high latency, why do short TCP transfers achieve a throughput well below the link bandwidth?
4. How does TCP react to packet loss, and how does this show up in your measurements? How does UDP react?
5. TCP signals the end of the transfer by closing the connection. Why can't UDP do the same, and what are the
   consequences for an application like ours?
6. If you had to measure the capacity of a real network path, would you use TCP or UDP? What would you need to add to our tools?

In [ ]:
# Clean up when you are done
net.stop()